# Benchmark of magnetic null finders on the Olshevsky et al. (2020) analytic field

This notebook samples the analytic magnetic field of Appendix A of Olshevsky et al.
(2020) on Cartesian grids and compares several null finders against the exact null
positions and types of their Table A.2:

* **new** — the `null_finder` package (version 0.1.0 and later);
* **local** — your local single-file `null_finder.py`;
* **Federica** — Federica's routine, through a small adapter (section 3).

For every implementation and resolution it reports which exact nulls are recovered,
position errors, types, spurious detections, agreement between implementations,
run times, the convergence of the position error with the grid spacing and a 3D view
of the detected nulls (without field lines).

**Note.** The published Table 1 reports the output of the trilinear method in MSAT.
The implementations compared here may use different search strategies, so at coarse
resolution they need not reproduce MSAT's failure modes. The positions of Table A.2 are
printed with 4–5 significant digits (|B| at the tabulated points is ~1e-4), so errors
below ~1e-4 are not meaningful.

In [ ]:
import importlib
import shutil
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

## 1. Configuration

Edit the paths below. Set an entry to `None` to leave that implementation out.

* `NEW_PKG_SRC`: the `src/` folder of the `null-finder` repository. With `None` the
  pip-installed package is used. The package is imported in a way that a local file
  called `null_finder.py` cannot shadow it.
* `LOCAL_NF_PATH`, `FEDERICA_PATH`: a `.py` file or a package folder.
* `RESOLUTIONS`: number of grid points per direction. 20 and 80 are those of Table 1.
  512³ needs about 4 GB of RAM in float64 for the field alone.

In [ ]:
# Repository src/ folder, or None for the pip-installed package.
NEW_PKG_SRC = None  # e.g. Path("~/codes/null-finder/src").expanduser()
# Other implementations to compare (None to skip), as a .py file or package folder.
LOCAL_NF_PATH = None  # e.g. Path("my_null_finder.py")
# Chiti (2020): https://github.com/FedericaChiti/Null_Finder
FEDERICA_PATH = None  # e.g. Path("Null_Finder/null_finder.py")
# Newton exit tolerance of Chiti's find_nulls (None -> its default, 1e-5).
FEDERICA_TOLERANCE = None

RESOLUTIONS = [20, 80, 512]
FIELD_DTYPE = np.float64
N_REPEAT_SMALL = 3  # timing: best of N_REPEAT_SMALL runs for grids up to 128^3

## 2. Loading the implementations

`import_new_package` imports the package even if the working directory contains a file
called `null_finder.py`.

`load_local_module` imports each local implementation under its own module name. A
single `.py` file is copied, with its timestamp, to `_bench_modules/<name>.py`, so:

* it does not collide with the package name;
* its Numba cache is kept separate;
* editing the original file is picked up on the next run.

Its original folder is also added to `sys.path`, so imports of sibling modules still work.

In [ ]:
BENCH_MODULE_DIR = Path("_bench_modules")


def import_new_package(src=None):
    # Import the null_finder *package*, never a local null_finder.py.
    for name in [
        m for m in sys.modules if m == "null_finder" or m.startswith("null_finder.")
    ]:
        del sys.modules[name]
    saved = sys.path[:]
    try:
        if src is not None:
            sys.path.insert(0, str(Path(src).expanduser().resolve()))
        else:
            cwd = Path.cwd().resolve()
            sys.path[:] = [
                p for p in sys.path if p not in ("", ".") and Path(p).resolve() != cwd
            ]
        importlib.invalidate_caches()
        mod = importlib.import_module("null_finder")
    finally:
        sys.path[:] = saved
    if not hasattr(mod, "__version__") or not hasattr(mod, "pad_periodic"):
        raise ImportError(
            f"'null_finder' resolved to {mod.__file__}, which is not the new package. "
            "Set NEW_PKG_SRC to the repository's src/ folder "
            "or pip-install the package."
        )
    return mod


def load_local_module(name, path):
    # Import a local implementation (file or package folder) as module `name`.
    path = Path(path).expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(path)
    sys.modules.pop(name, None)
    if path.is_dir():
        if path.name == "null_finder":
            raise ImportError(
                f"{path} is a package called null_finder: rename it or copy it."
            )
        if str(path.parent) not in sys.path:
            sys.path.append(str(path.parent))
        importlib.invalidate_caches()
        return importlib.import_module(path.name)
    BENCH_MODULE_DIR.mkdir(exist_ok=True)
    target = BENCH_MODULE_DIR / f"{name}.py"
    if not target.exists() or target.stat().st_mtime != path.stat().st_mtime:
        shutil.copy2(path, target)
    for p in (str(BENCH_MODULE_DIR.resolve()), str(path.parent)):
        if p not in sys.path:
            sys.path.insert(
                0 if p == str(BENCH_MODULE_DIR.resolve()) else len(sys.path), p
            )
    importlib.invalidate_caches()
    return importlib.import_module(name)


nf_new = import_new_package(NEW_PKG_SRC)
print(f"new      : null_finder {nf_new.__version__}  ({Path(nf_new.__file__).parent})")
nf_local = load_local_module("nf_local", LOCAL_NF_PATH) if LOCAL_NF_PATH else None
if nf_local is not None:
    print(f"local    : {LOCAL_NF_PATH}")
nf_fede = load_local_module("nf_federica", FEDERICA_PATH) if FEDERICA_PATH else None
if nf_fede is not None:
    print(f"Federica : {FEDERICA_PATH}")

## 3. A common interface for all implementations

Each runner takes `(bx, by, bz, x, y, z)` and returns a dictionary with:

* `pos`: null positions, `(N, 3)`;
* `type`: type names, among `"A"`, `"B"`, `"As"`, `"Bs"` and `"unknown"`;
* `charge`: topological charge (±1), or NaN if not provided;
* `raw`: the native output of the routine.

`run_null_finder_api` covers every implementation that follows the `null_finder` API
(`find_magnetic_nulls(...)` returning `pos`, `null_type`, `null_charge`).

`run_federica` wraps `find_nulls(nx, ny, nz, xv, yv, zv, Bx, By, Bz, tolerance)` of
Federica Chiti's routine (Chiti, Pontin & Scott). That routine returns positions only,
so the notebook computes the type and the charge of each of its nulls:

* the Jacobian comes from the routine's own trilinear interpolant of the cell, through its
  functions `trilinear_coeffs` and `jacobian`;
* the classification rules (`classify_jacobian`) are the same as in the `null_finder`
  package.

Type differences against the other implementations therefore reflect where the nulls
are, not different classification rules.

Behaviour of `find_nulls` to keep in mind when reading the results:

* one Newton start per cell (the centre), with at most 11 iterations;
* a bilinear face test before Newton;
* nulls with z > `zv[-2]`, i.e. in the last layer of cells, are discarded on purpose
  (none of the nulls of this benchmark is there);
* a null found from two neighbouring cells is reported twice (it would appear as
  *spurious*).

It is pure Python/NumPy, so it needs no warm-up.

In [ ]:
TYPE_CODES = {0: "unknown", 1: "A", 2: "B", 3: "As", 4: "Bs"}
TYPE_ALIASES = {
    "a": "A",
    "ar": "A",
    "a-radial": "A",
    "radial a": "A",
    "negative": "A",
    "b": "B",
    "br": "B",
    "b-radial": "B",
    "radial b": "B",
    "positive": "B",
    "as": "As",
    "a-spiral": "As",
    "spiral a": "As",
    "bs": "Bs",
    "b-spiral": "Bs",
    "spiral b": "Bs",
}
TYPE_CHARGE = {"A": 1, "As": 1, "B": -1, "Bs": -1}


def normalize_type(t):
    if isinstance(t, (int, np.integer)):
        return TYPE_CODES.get(int(t), "unknown")
    return TYPE_ALIASES.get(str(t).strip().lower(), "unknown")


def standardize(pos, types, charge=None, raw=None):
    pos = np.asarray(pos, dtype=float).reshape(-1, 3)
    types = [normalize_type(t) for t in types]
    if len(types) != len(pos):
        raise ValueError("pos and types must have the same length")
    charge = np.full(len(pos), np.nan) if charge is None else np.asarray(charge, float)
    return {"pos": pos, "type": types, "charge": charge, "raw": raw}


def run_null_finder_api(mod, bx, by, bz, x, y, z, **kw):
    res = mod.find_magnetic_nulls(bx, by, bz, x, y, z, verbose=False, **kw)
    return standardize(res["pos"], res["null_type"], res.get("null_charge"), raw=res)


def classify_jacobian(J, rel_tol=1e-10):
    # Same rules as null_finder: a clear 2+1 eigenvalue structure, i.e. fan
    # eigenvalues (a real pair, or a complex pair for spiral nulls) whose real parts
    # share one sign, and a real spine eigenvalue of the opposite sign.
    # Returns (type name, charge).
    J = np.asarray(J, dtype=float)
    charge = int(np.sign(np.linalg.det(J)))
    ev = np.linalg.eigvals(J)
    tol = rel_tol * np.max(np.abs(ev))
    cplx = np.abs(ev.imag) > tol
    if cplx.sum() == 2:
        spine_re, fan_re, spiral = ev.real[~cplx][0], ev.real[cplx][0], True
    elif cplx.sum() == 0:
        signs = np.where(ev.real > tol, 1, np.where(ev.real < -tol, -1, 0))
        if np.any(signs == 0) or abs(signs.sum()) != 1:
            return "unknown", charge
        fan_sign = signs.sum()
        spine_re = ev.real[signs == -fan_sign][0]
        fan_re = ev.real[signs == fan_sign][0]
        spiral = False
    else:
        return "unknown", charge
    if fan_re < -tol and spine_re > tol:
        return ("As" if spiral else "A"), charge
    if fan_re > tol and spine_re < -tol:
        return ("Bs" if spiral else "B"), charge
    return "unknown", charge


def run_federica(mod, bx, by, bz, x, y, z, tolerance=FEDERICA_TOLERANCE):
    nulls = mod.find_nulls(
        len(x), len(y), len(z), x, y, z, bx, by, bz, tolerance=tolerance
    )
    pos = np.array([[n["x"], n["y"], n["z"]] for n in nulls], dtype=float).reshape(
        -1, 3
    )
    types, charges = [], []
    for n in nulls:
        coeffs = np.array(
            [
                mod.trilinear_coeffs(x, y, z, n["i"], n["j"], n["k"], b)
                for b in (bx, by, bz)
            ]
        )
        t, q = classify_jacobian(mod.jacobian(n["x"], n["y"], n["z"], coeffs))
        types.append(t)
        charges.append(q)
    return standardize(pos, types, charges, raw=nulls)


# name -> runner and whether it needs a warm-up call (Numba compilation) before timing
FINDERS = {"new": {"run": lambda *a: run_null_finder_api(nf_new, *a), "warmup": True}}
if nf_local is not None:
    FINDERS["local"] = {
        "run": lambda *a: run_null_finder_api(nf_local, *a),
        "warmup": True,
    }
if nf_fede is not None:
    FINDERS["Federica"] = {"run": lambda *a: run_federica(nf_fede, *a), "warmup": False}
print("Implementations:", list(FINDERS))

## 4. Analytic benchmark field

Eq. (A.1) of the paper: a triply periodic background, a small analytic perturbation and
the curls of seven localized Gaussian vector potentials (flux rings).

* For A = φ ê_z: curl A = (∂φ/∂y, −∂φ/∂x, 0).
* For A = φ ê_y: curl A = (−∂φ/∂z, 0, ∂φ/∂x).
* For A = φ ê_x: curl A = (0, ∂φ/∂z, −∂φ/∂y).

Two corrections with respect to the printed text:

* **Y₂ = 0.117 instead of 0.0117.** With the value as printed in Table A.1, the field does
  not vanish at the tabulated nulls 10 and 12. The residual is dominated by the second
  flux ring, and with 0.117 the tabulated nulls are consistent with the field to the
  printed precision.
* **√30 / 7**, not √(30/7), in the constant of B_y.

The implementation is identical to the straightforward one (differences ~1e-14), but it
works with sparse meshgrids and separable Gaussians, so it never builds the full X, Y, Z
arrays. For 512³ that is about 2.7× less memory. The arrays are `(nx, ny, nz)` and
Fortran-contiguous.

In [ ]:
pi = np.pi

# Table A.1 parameters: a_i, k_i, l_i, X_i, Y_i, Z_i
# (Y_2 corrected from 0.0117 to 0.117, see above)
PARAMS_A1 = np.array(
    [
        [-0.2, 50.0, 0.2, 2.8455, 0.0267, -0.147],
        [-0.3, 4.0, 0.2, np.pi + 0.191, 0.117, np.pi],
        [1.0, 0.8, 1.0, 0.1, np.pi - 0.037, 0.2],
        [-0.22, 1.0, 0.9, np.pi + 0.0771, np.pi + 0.087, np.pi],
        [-0.22, 2.0, 0.5, np.pi / 2, np.pi / 2, np.pi / 2],
        [1.0, 1.0, 1.0, 0.073, 0.0198, -0.07],
        [-0.22, 2.0, 0.5, np.pi / 2, np.pi / 2, np.pi / 2],
    ],
    dtype=float,
)


def olshevsky2020_field(x, y, z, dtype=np.float64):
    X, Y, Z = np.meshgrid(
        np.asarray(x, float),
        np.asarray(y, float),
        np.asarray(z, float),
        indexing="ij",
        sparse=True,
    )
    shape = (X.shape[0], Y.shape[1], Z.shape[2])
    bx = np.zeros(shape, dtype=np.float64, order="F")
    by = np.zeros(shape, dtype=np.float64, order="F")
    bz = np.zeros(shape, dtype=np.float64, order="F")
    bx += -2.0 * np.sin(2.0 * Y) + np.sin(Z)
    by += 2.0 * np.sin(X) + np.sin(Z)
    bz += np.sin(X) + np.sin(Y)
    bx += 0.04 * (-2.0 * np.sin(2.0 * Y) + np.sin(Y) + 0.1195)
    bx += 0.04 * 2.0 * np.sin(X) * np.cos(Z)
    by += 0.04 * (2.0 * np.sin(X - Z) + np.sin(X + Z) + np.sqrt(30.0) / 7.0)
    bz += 0.04 * (np.sin(Y) + 2.0 * np.sin(2.0 * Y) - 0.1378162)
    bz += 0.04 * (-2.0) * np.cos(X) * np.sin(Z)
    for idx, (a, kappa, ell, Xi, Yi, Zi) in enumerate(PARAMS_A1, start=1):
        amp = a * kappa
        gx = np.exp(-((X - Xi) ** 2) / (a**2 if idx <= 5 else ell**2))
        gy = np.exp(-((Y - Yi) ** 2) / (a**2 if idx <= 2 or idx > 5 else ell**2))
        gz = np.exp(-((Z - Zi) ** 2) / (ell**2 if idx <= 2 else a**2))
        c = 2.0 * amp / a**2
        if idx <= 2:
            bx += (gx * (-c * (Y - Yi) * gy)) * gz
            by += ((c * (X - Xi) * gx) * gy) * gz
        elif idx <= 5:
            bx += (gx * gy) * (c * (Z - Zi) * gz)
            bz += ((-c * (X - Xi) * gx) * gy) * gz
        else:
            by += (gx * gy) * (-c * (Z - Zi) * gz)
            bz += (gx * (c * (Y - Yi) * gy)) * gz
    return tuple(
        b if b.dtype == dtype else np.asfortranarray(b.astype(dtype))
        for b in (bx, by, bz)
    )


def create_bfield(N, dtype=FIELD_DTYPE):
    c = np.linspace(-np.pi / 2, 3 * np.pi / 2, N)
    return (c, c.copy(), c.copy(), *olshevsky2020_field(c, c, c, dtype=dtype))

## 5. Exact nulls (Table A.2) and field residual at the tabulated positions

In [ ]:
EXACT_NULLS = pd.DataFrame(
    {
        "null_id": np.arange(1, 15),
        "exact_type": [
            "Bs",
            "As",
            "A",
            "Bs",
            "As",
            "Bs",
            "Bs",
            "A",
            "As",
            "Bs",
            "As",
            "As",
            "Bs",
            "Bs",
        ],
        "x_exact": [
            0.2952,
            0.04581,
            2.8657,
            2.9737,
            3.2286,
            -0.2124,
            -0.2728,
            3.1391,
            0.01951,
            3.3225,
            -0.0017705,
            3.3119,
            2.9915,
            3.0203,
        ],
        "y_exact": [
            -0.2505,
            -0.0005391,
            -0.1878,
            -0.1500,
            0.06090,
            3.3212,
            0.2386,
            3.1344,
            -0.01637,
            0.1453,
            3.1360,
            0.1539,
            -0.09684,
            3.0182,
        ],
        "z_exact": [
            -1.4803,
            -0.01064,
            -0.5961,
            0.2246,
            0.2457,
            0.4444,
            1.2466,
            -0.03510,
            3.2123,
            2.9898,
            3.1707,
            3.2509,
            3.5047,
            3.3935,
        ],
    }
)
exact_pos = EXACT_NULLS[["x_exact", "y_exact", "z_exact"]].to_numpy(float)


def eval_field_at_points(points):
    out = []
    for p in np.asarray(points, dtype=float):
        b = olshevsky2020_field(p[:1], p[1:2], p[2:3], dtype=np.float64)
        out.append([b[0][0, 0, 0], b[1][0, 0, 0], b[2][0, 0, 0]])
    return np.asarray(out)


B_exact = eval_field_at_points(exact_pos)
check = EXACT_NULLS.copy()
check[["Bx", "By", "Bz"]] = B_exact
check["|B|"] = np.linalg.norm(B_exact, axis=1)
check

## 6. Run all implementations

The first call of a Numba routine compiles it, so the Numba-based implementations
(`warmup=True`) are run once on a small grid before timing; Federica's routine needs no
warm-up. For grids up to 128³ the time is the best of
`N_REPEAT_SMALL` runs. Field construction is not included in the times. If an
implementation fails, the error is recorded and the others continue.

In [ ]:
warm = create_bfield(12)
for name, spec in FINDERS.items():
    if not spec["warmup"]:
        continue
    t0 = time.perf_counter()
    try:
        spec["run"](*warm[3:], *warm[:3])
        print(f"{name:10s} compiled / warmed up in {time.perf_counter() - t0:6.1f} s")
    except Exception as e:
        print(f"{name:10s} FAILED on warm-up: {type(e).__name__}: {e}")

RESULTS = {}  # (name, N) -> standardized result
TIMES = {}  # (name, N) -> seconds
ERRORS = {}  # (name, N) -> message
GRIDS = {}  # N -> grid spacing

for N in RESOLUTIONS:
    t0 = time.perf_counter()
    x, y, z, bx, by, bz = create_bfield(N)
    GRIDS[N] = x[1] - x[0]
    t_field = time.perf_counter() - t0
    print(f"\nN = {N}: field built in {t_field:.2f} s (h = {GRIDS[N]:.4f})")
    nrep = N_REPEAT_SMALL if N <= 128 else 1
    for name, spec in FINDERS.items():
        try:
            best = np.inf
            for _ in range(nrep):
                t0 = time.perf_counter()
                out = spec["run"](bx, by, bz, x, y, z)
                best = min(best, time.perf_counter() - t0)
            RESULTS[name, N], TIMES[name, N] = out, best
            print(f"  {name:10s} {len(out['pos']):3d} nulls  {best:8.4f} s")
        except Exception as e:
            ERRORS[name, N] = f"{type(e).__name__}: {e}"
            print(f"  {name:10s} FAILED: {ERRORS[name, N]}")
    del bx, by, bz

## 7. Comparison with the exact nulls

Detected nulls are assigned one-to-one to the exact ones by minimum total distance
(Hungarian algorithm).

* An exact null counts as **recovered** if its assigned null lies within one grid spacing h.
* Detected nulls that are not assigned within h are **spurious**.
* "type ok" compares the detected type with Table A.2.
* "charge ok" compares the detected charge with the charge implied by the exact type
  (A/As: +1, B/Bs: −1).

In [ ]:
def match_to_exact(out, h):
    found = out["pos"]
    m = pd.DataFrame(index=EXACT_NULLS.index)
    m["type"], m["err"], m["charge"], m["idx"] = "—", np.nan, np.nan, -1
    if len(found) == 0:
        return m, 0
    D = np.linalg.norm(exact_pos[:, None, :] - found[None, :, :], axis=-1)
    rows, cols = linear_sum_assignment(D)
    for r, c in zip(rows, cols, strict=True):
        if D[r, c] <= h:
            m.loc[r, ["type", "err", "charge", "idx"]] = (
                out["type"][c],
                D[r, c],
                out["charge"][c],
                c,
            )
    n_spurious = len(found) - int((m["idx"] >= 0).sum())
    return m, n_spurious


MATCHES, SPURIOUS = {}, {}
for key, out in RESULTS.items():
    MATCHES[key], SPURIOUS[key] = match_to_exact(out, GRIDS[key[1]])

exact_charge = EXACT_NULLS["exact_type"].map(TYPE_CHARGE).to_numpy(float)
rows = []
for (name, N), m in MATCHES.items():
    ok = m["idx"] >= 0
    rows.append(
        {
            "implementation": name,
            "N": N,
            "h": GRIDS[N],
            "found": len(RESULTS[name, N]["pos"]),
            "recovered (of 14)": int(ok.sum()),
            "type ok": int((m["type"] == EXACT_NULLS["exact_type"]).sum()),
            "charge ok": int((m["charge"].to_numpy() == exact_charge).sum())
            if ok.any()
            else 0,
            "spurious": SPURIOUS[name, N],
            "median err": m.loc[ok, "err"].median(),
            "max err": m.loc[ok, "err"].max(),
            "max err / h": m.loc[ok, "err"].max() / GRIDS[N],
            "time [s]": TIMES[name, N],
        }
    )
for (name, N), msg in ERRORS.items():
    rows.append(
        {"implementation": name, "N": N, "h": GRIDS.get(N, np.nan), "error": msg}
    )

SUMMARY = pd.DataFrame(rows).sort_values(["N", "implementation"]).reset_index(drop=True)
SUMMARY.round(4)

### Per-null detail

A type shown as `X≠Y` means the implementation found type X where Table A.2 lists Y.
`—` means the exact null was not recovered within one grid spacing. Federica's types are
computed by the notebook from her own trilinear Jacobian (section 3).

In [ ]:
def per_null_table(N):
    t = EXACT_NULLS[["null_id", "exact_type"]].copy()
    for name in FINDERS:
        if (name, N) not in MATCHES:
            continue
        m = MATCHES[name, N]
        t[f"type {name}"] = [
            ft if ft in ("—", et) else f"{ft}≠{et}"
            for ft, et in zip(m["type"], EXACT_NULLS["exact_type"], strict=True)
        ]
        t[f"err {name}"] = m["err"]
    return t


for N in RESOLUTIONS:
    print(f"N = {N}  (h = {GRIDS.get(N, np.nan):.4f})")
    display(per_null_table(N).round(5))

## 8. Agreement between implementations

For each pair of implementations and each resolution, the detected nulls are matched
one-to-one, within one grid spacing. The trilinear interpolant of the grid has exactly
defined nulls, so two correct trilinear finders should agree up to their convergence
tolerance. Larger differences point to a different interpolation, a missed root or a
looser tolerance.

In [ ]:
pair_rows = []
names = list(FINDERS)
for N in RESOLUTIONS:
    for i, a in enumerate(names):
        for b in names[i + 1 :]:
            if (a, N) not in RESULTS or (b, N) not in RESULTS:
                continue
            pa, pb = RESULTS[a, N]["pos"], RESULTS[b, N]["pos"]
            if len(pa) == 0 or len(pb) == 0:
                continue
            D = np.linalg.norm(pa[:, None] - pb[None], axis=-1)
            r, c = linear_sum_assignment(D)
            keep = D[r, c] <= GRIDS[N]
            r, c = r[keep], c[keep]
            ta = np.array(RESULTS[a, N]["type"])[r]
            tb = np.array(RESULTS[b, N]["type"])[c]
            dmax = D[r, c].max() if len(r) else np.nan
            pair_rows.append(
                {
                    "N": N,
                    "A": a,
                    "B": b,
                    "n A": len(pa),
                    "n B": len(pb),
                    "common": len(r),
                    "only A": len(pa) - len(r),
                    "only B": len(pb) - len(r),
                    "max |Δpos|": f"{dmax:.2e}",
                    "max |Δpos| / h": f"{dmax / GRIDS[N]:.2e}",
                    "type disagreements": int((ta != tb).sum()),
                }
            )
AGREEMENT = pd.DataFrame(pair_rows)
AGREEMENT

## 9. Convergence of the position error

For a smooth field the nulls of the trilinear interpolant converge as O(h²). The error
saturates at about 1e-4 because of the printed precision of Table A.2.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
# Decreasing marker sizes, so that coinciding results remain visible.
for k, name in enumerate(FINDERS):
    Ns = [N for N in RESOLUTIONS if (name, N) in MATCHES]
    if not Ns:
        continue
    h = np.array([GRIDS[N] for N in Ns])
    med = [MATCHES[name, N]["err"].median() for N in Ns]
    mx = [MATCHES[name, N]["err"].max() for N in Ns]
    ms = 10 - 3 * k
    (line,) = ax.loglog(h, med, "o-", ms=ms, label=f"{name} (median)")
    ax.loglog(
        h, mx, "s--", ms=ms, color=line.get_color(), alpha=0.7, label=f"{name} (max)"
    )
hh = np.array(sorted(GRIDS.values()))
anchor = [
    MATCHES[n, max(GRIDS, key=GRIDS.get)]["err"].median()
    for n in FINDERS
    if (n, max(GRIDS, key=GRIDS.get)) in MATCHES
]
if len(hh) and anchor:
    ax.loglog(hh, anchor[0] * (hh / hh.max()) ** 2, "k:", label="∝ h²")
ax.axhline(1e-4, color="0.6", lw=0.8)
ax.text(
    hh.max() if len(hh) else 0.3,
    1.15e-4,
    "Table A.2 precision",
    color="0.4",
    fontsize=8,
    ha="right",
    va="bottom",
)
ax.set_xlabel("grid spacing h")
ax.set_ylabel("distance to exact null")
ax.legend(fontsize=8)
ax.grid(True, which="both", alpha=0.3)
plt.show()

## 10. 3D view of the detected nulls

The view uses the finest resolution. Exact nulls are drawn as black circles and the
detections of each implementation as coloured markers; no field lines are drawn.

In [ ]:
N_view = max(RESOLUTIONS)
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(projection="3d")
ax.scatter(
    *exact_pos.T,
    s=140,
    facecolors="none",
    edgecolors="k",
    linewidths=1.5,
    label="exact (Table A.2)",
)
for i, row in EXACT_NULLS.iterrows():
    ax.text(
        exact_pos[i, 0] + 0.12,
        exact_pos[i, 1],
        exact_pos[i, 2],
        f"{row.null_id} {row.exact_type}",
        fontsize=7,
    )
# Decreasing marker sizes, so that coinciding detections remain visible.
for k, (name, mk) in enumerate(zip(FINDERS, ["o", "^", "s", "D"], strict=False)):
    if (name, N_view) in RESULTS:
        p = RESULTS[name, N_view]["pos"]
        ax.scatter(
            *p.T, s=60 / (k + 1), marker=mk, alpha=0.8, label=f"{name} ({len(p)})"
        )
lim = (-np.pi / 2, 3 * np.pi / 2)
ax.set(
    xlim=lim,
    ylim=lim,
    zlim=lim,
    xlabel="x",
    ylabel="y",
    zlabel="z",
    title=f"Detected nulls, N = {N_view}",
)
ax.legend(fontsize=8, loc="upper left")
plt.show()